<a href="https://colab.research.google.com/github/bsahasra11/ReAct-agent/blob/main/Copy_of_agent_loop_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [1]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

Paste your Gemini API key and press Enter: ··········
API key set.


In [2]:
# Quiet ensures your UI is not full of logs.
!pip install google-generativeai --quiet
print("Done.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-spanner 3.71.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-storage-control 1.17.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-language 2.22.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-translate 3.28.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-bigquery-storage 2.43.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
tensorflow 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 5.29.6 which is incompatible.
google-cloud-functions 1.25.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
grpc-google-iam

In [3]:
# @title User input needed.

# TODO: Fill in documents in the format mentioned in the first example.
RECIPES = {
    "spaghetti_carbonara": (
        "8 oz spaghetti pasta"
        "4 oz guanciale or pancetta diced"
        "2 large eggs plus 2 egg yolks"
        "1 cup freshly grated Pecorino Romano cheese"
        "Freshly cracked black pepper to taste"
        "Bring a large pot of salted water to a boil and cook spaghetti until al dente."
        "Sauté diced guanciale in a skillet over medium heat until crispy, then remove from heat."
        "Whisk eggs, egg yolks, and grated Pecorino Romano together in a bowl until smooth."
        "Toss drained pasta into the skillet with guanciale, remove from heat, and quickly stir in egg mixture."
        "Add pasta water as needed to create a glossy sauce and finish with black pepper."
    ),
    "margherita_pizza": (
        "1 ball prepared pizza dough"
        "1/2 cup crushed San Marzano tomatoes"
        "4 oz fresh mozzarella sliced"
        "Fresh basil leaves"
        "1 tbsp extra virgin olive oil"
        "Preheat oven with a pizza stone inside to 500°F (260°C)."
        "Stretch pizza dough into a 12-inch circle on parchment paper."
        "Spread crushed tomatoes evenly over the base leaving a 1-inch border."
        "Arrange fresh mozzarella slices across the tomato sauce."
        "Bake for 8-10 minutes until crust is charred and cheese is bubbly, then top with fresh basil and olive oil."
    ),
    "classic_pancakes": (
        "1.5 cups all-purpose flour"
        "3.5 tsp baking powder"
        "1 tbsp granulated sugar"
        "1.25 cups milk"
        "1 large egg"
        "3 tbsp melted butter"
        "Sift flour, baking powder, and sugar together in a large bowl."
        "Whisk milk, egg, and melted butter in a separate container."
        "Combine wet and dry ingredients until just mixed, leaving small lumps."
        "Pour 1/4 cup batter onto a greased hot griddle, flip when bubbles surface, and cook until golden."
    ),
    "guacamole": (
        "3 ripe avocados mashed"
        "1/2 cup finely diced red onion"
        "1/2 cup fresh cilantro chopped"
        "1 jalapeño seeded and minced"
        "2 tbsp fresh lime juice"
        "1/2 tsp sea salt"
        "Slice avocados in half, remove pits, and scoop flesh into a mixing bowl."
        "Mash avocados with a fork until desired texture is reached."
        "Stir in red onion, cilantro, jalapeño, lime juice, and sea salt."
        "Taste and adjust salt or lime juice before serving with tortilla chips."
    )
}

In [4]:
# @title Define a tool
def search_recipes(query: str) -> str:
    """Search the recipe collection for details relevant to the query."""
    query_lower = query.lower()
    results = []

    for recipe_name, content in RECIPES.items():
        words = set(query_lower.split())
        match_count = sum(1 for word in words if word in content.lower())
        if match_count > 0:
            results.append((match_count, recipe_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant recipes found for that query."

    # Return the top two matches
    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")
    return "\n\n".join(output)


# Sanity check for recipe search.
print(search_recipes("spaghetti"))

[Source: spaghetti_carbonara]
8 oz spaghetti pasta4 oz guanciale or pancetta diced2 large eggs plus 2 egg yolks1 cup freshly grated Pecorino Romano cheeseFreshly cracked black pepper to tasteBring a large pot of salted water to a boil and cook spaghetti until al dente.Sauté diced guanciale in a skillet over medium heat until crispy, then remove from heat.Whisk eggs, egg yolks, and grated Pecorino Romano together in a bowl until smooth.Toss drained pasta into the skillet with guanciale, remove from heat, and quickly stir in egg mixture.Add pasta water as needed to create a glossy sauce and finish with black pepper.


In [5]:
# @title Ask gemini to search for something
import google.generativeai as genai
import time

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

model = genai.GenerativeModel(
    # This will break.
    # TODO: Find the right model to call in this cell from developer docs.
    model_name="gemini-3.8-flash",
    tools=[search_recipes],
    system_instruction=(
        '''
        You are a chef training at a Michelin star restaurant.
        When you need specific information about temperatures,
        ingredients to use, cook time etc. use the search_documents tool.
        Only answer based on what you find in the documents.
        If the documents do not contain the answer, say so clearly.
        '''
    )
)

print("Model ready:", model.model_name)

/tmp/ipykernel_2161/68343949.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Model ready: models/gemini-3.8-flash


In [6]:
import time
from google import genai
from google.genai import types

client = genai.Client()  # reads GEMINI_API_KEY / GOOGLE_API_KEY from env

search_tool = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="search_recipes",
        description="Search the recipe collection to find the most relevant recipe details.",
        parameters={
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    )
])

config = types.GenerateContentConfig(
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)


def send_with_retry(contents, max_retries=3):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=model.model_name,  # This will now correctly resolve to "models/gemini-3.8-flash"
                contents=contents, config=config
            )
        except Exception as e:
            if "429" in str(e) and attempt < max_retries - 1:
                wait = 20 * (attempt + 1)
                print(f"  Rate limit hit, waiting {wait}s...")
                time.sleep(wait)
            else:
                raise


def run_agent(question: str, max_steps: int = 8) -> str:
    print(f"Question: {question}\n" + "-" * 50)
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)
        model_content = response.candidates[0].content

        # Append the model's turn UNMODIFIED, so thought signatures are preserved
        contents.append(model_content)

        calls = [p.function_call for p in model_content.parts if p.function_call]
        if not calls:
            print(f"\nFinal answer:\n{response.text}")
            return response.text

        # Run every requested tool call, reply with all results in ONE user turn
        result_parts = []
        for call in calls:
            args = dict(call.args)
            print(f"Step {step}: model calls {call.name}({args})")
            result = search_recipes(**args)
            print(f"  Result preview: {str(result)[:100]}...")
            result_parts.append(
                types.Part.from_function_response(
                    name=call.name, response={"result": result}
                )
            )
        contents.append(types.Content(role="user", parts=result_parts))

    return "(Stopped: hit max_steps)"


_ = run_agent("What are the ingredients for spaghetti carbonara?")

Question: What are the ingredients for spaghetti carbonara?
--------------------------------------------------
Step 1: model calls search_recipes({'query': 'spaghetti carbonara'})
  Result preview: [Source: spaghetti_carbonara]
8 oz spaghetti pasta4 oz guanciale or pancetta diced2 large eggs plus ...

Final answer:
The ingredients for traditional spaghetti carbonara are:

* **Spaghetti:** 8 oz
* **Guanciale** (or pancetta): 4 oz, diced
* **Eggs:** 2 large whole eggs plus 2 egg yolks
* **Pecorino Romano cheese:** 1 cup, freshly grated
* **Black pepper:** Freshly cracked, to taste
* **Salt & pasta water:** Salt for boiling the pasta water, with reserved starchy cooking water to emulsify the sauce


In [8]:
# @title Ask follow up questions

# TODO: Enter questions based on documents for the agent to answer.
questions = [
    "Add a question here.",
    # This question is not answered by the database.
    "How many ghosts eat carrots everyday?",
    # This question is part of the general knowledge of the model.
    "what ingredients are used in guacamole?",
    "what are the ingredients in margherita pizza?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # stay within the free-tier rate limit.
    print("\n" + "=" * 60)
    run_agent(q)



Question: Add a question here.
--------------------------------------------------

Final answer:
It looks like you might have pasted a placeholder! 

What question or recipe can I help you with today?

Question: How many ghosts eat carrots everyday?
--------------------------------------------------

Final answer:
Probably **zero**! 

Aside from ghosts being mythical, they’re incorporeal—so any carrot they tried to eat would fall right through them. Plus, most folklore suggests that if ghosts eat anything at all, they stick to **boo-berries**, **ghoulash**, or **spook-ghetti**!

Question: what ingredients are used in guacamole?
--------------------------------------------------


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [11]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)

AssertionError: 